# CourtListener Keyword vs. Semantic Search

This notebook runs the same case-name query twice through `CourtListenerClient.search`: once with the default keyword search and once with `semantic=True`. It displays the returned count, exact-name rank, ranking scores, citations, and the first 20 candidates.

Keep `CASE_NAME_QUERIES` short: each name makes two CourtListener API requests and the configured account rate limits still apply.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
from IPython.display import display

from mellea_lrc.core.env import load_env_file
from mellea_lrc.courtlistener import CourtListenerClient

load_env_file(Path("../.env"), override=True)

# Start with one query to stay comfortably inside CourtListener's rate limits.
CASE_NAME_QUERIES = ["Johnson v. City of Shelby"]
SEARCH_TYPE = "o"  # Case-law opinion clusters; semantic search only supports case law.

client = CourtListenerClient()
pd.set_option("display.max_colwidth", 100)

In [ ]:
def normalize_case_name(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", value.casefold()).strip()


def search_rows(query: str, *, semantic: bool) -> list[dict[str, object]]:
    response = client.search(query, SEARCH_TYPE, semantic=semantic)
    raw = response["raw"]
    results = raw.get("results", []) if isinstance(raw, dict) else []
    total_count = raw.get("count") if isinstance(raw, dict) else None
    mode = "semantic" if semantic else "keyword"
    expected = normalize_case_name(query)

    rows = []
    for rank, result in enumerate(results, start=1):
        case_name = str(result.get("caseName") or result.get("case_name") or "")
        scores = result.get("meta", {}).get("score", {})
        rows.append(
            {
                "query": query,
                "mode": mode,
                "total_count": total_count,
                "rank": rank,
                "exact_name": normalize_case_name(case_name) == expected,
                "case_name": case_name,
                "court_id": result.get("court_id") or result.get("court"),
                "date_filed": result.get("dateFiled") or result.get("date_filed"),
                "citations": result.get("citation"),
                "bm25_score": scores.get("bm25"),
                "semantic_score": scores.get("semantic"),
                "absolute_url": result.get("absolute_url"),
            }
        )
    return rows

In [ ]:
rows = []
for query in CASE_NAME_QUERIES:
    rows.extend(search_rows(query, semantic=False))
    rows.extend(search_rows(query, semantic=True))

results = pd.DataFrame(rows)

summary_rows = []
for (query, mode), group in results.groupby(["query", "mode"], sort=False):
    exact_ranks = group.loc[group["exact_name"], "rank"].tolist()
    summary_rows.append(
        {
            "query": query,
            "mode": mode,
            "reported_count": group["total_count"].iloc[0],
            "returned_results": len(group),
            "first_exact_rank": exact_ranks[0] if exact_ranks else None,
        }
    )

display(pd.DataFrame(summary_rows))
display(
    results[
        [
            "mode",
            "rank",
            "exact_name",
            "case_name",
            "court_id",
            "date_filed",
            "citations",
            "bm25_score",
            "semantic_score",
            "absolute_url",
        ]
    ]
)